In [20]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import os, json, torch, gc

In [21]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", device)

Device: mps


In [22]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype="auto", device_map="auto"
)

Loading weights: 100%|██████████| 824/824 [00:11<00:00, 73.81it/s] 


In [23]:
IMG_DIR = "/Users/administrator/Documents/SpottySearch/Pottery pictures"
CACHE = "captions.json"

MAX_PIXELS = 1024 * 1024 

image_files = [f for f in os.listdir('Pottery pictures') if f.lower().endswith('.jpg')]
image_files.sort()

In [24]:
captions = {}
if os.path.exists(CACHE):
    with open(CACHE) as f:
        captions = json.load(f)

In [25]:
PROMPT = (
    "List every distinct pottery item in this image. For each, give its colour, "
    "form (mug, plate, bowl, figurine, etc.) and any patterns or designs, "
    "one short sentence per item."
)

In [26]:
def caption_image(path):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": path, "max_pixels": MAX_PIXELS},
            {"type": "text", "text": PROMPT},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to("mps")
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    cap = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()
    del inputs, generated_ids, trimmed, image_inputs
    gc.collect()
    torch.mps.empty_cache()
    return cap

In [ ]:
for i, file in enumerate(image_files):
    if file in captions:                   # skip what's already done
        continue
    captions[file] = caption_image(os.path.join(IMG_DIR, file))
    with open(CACHE, "w") as f:            # save after each one
        json.dump(captions, f, indent=2)
    print(f"[{i+1}/{len(image_files)}] {file}: {captions[file][:80]}")

[1/43] PHOTO-2026-06-02-06-58-12 10.jpg: 1. **Plate**: White with floral designs, including yellow flowers and green leav
[2/43] PHOTO-2026-06-02-06-58-12 11.jpg: 1. **Plate**: Light green with a checkered pattern of pink and white squares.
2.
[3/43] PHOTO-2026-06-02-06-58-12 12.jpg: 1. **Figurine**: Blue and green, shaped like a rabbit with a pointed hat.
2. **P
[4/43] PHOTO-2026-06-02-06-58-12 13.jpg: 1. **Plate**: Pink with small pink hearts arranged in vertical lines.
2. **Figur
